In [2]:
import pandas as pd
import numpy as np

In [3]:
validator_metadata = pd.read_csv('../int/validator_metadata_v2.csv')
largest_pools = ('Lido','Coinbase','Binance','Rocketpool','Kraken','OKX','Bitcoin Suisse','Ledger Live','Ether.Fi','Mantle')
validator_metadata.loc[~validator_metadata['pool'].isin(largest_pools), 'pool'] = 'Other Stakers'

/var/folders/mb/5hm6pgrs3zj_1m_kgvpt40jw0000gn/T/ipykernel_464/176968062.py:1: DtypeWarning: Columns (9) have mixed types. Specify dtype option on import or set low_memory=False.
  validator_metadata = pd.read_csv('../int/validator_metadata_v2.csv')


In [3]:
print(len(validator_metadata['pool'].unique()))

11


In [4]:
validator_exits = validator_metadata[['validator_index', 'exit_epoch']].dropna(subset=['exit_epoch'])
validator_exits['slot'] = validator_exits['exit_epoch'] * 32
validator_exits['exits'] = 1
validator_exits = validator_exits[validator_exits['slot'] <= 8986176]
validator_exits = validator_exits.drop(columns=['exit_epoch'])
validator_exits = pd.merge(validator_exits, validator_metadata[['validator_index', 'pool', 'category', 'pool_size_label']], on='validator_index', how='left')
validator_exits = validator_exits.sort_values('slot')

# Calculate cumulative exits for each pool
validator_exits['pool_cumulative'] = validator_exits.groupby('pool')['exits'].cumsum()

# Calculate cumulative exits for each category
validator_exits['category_cumulative'] = validator_exits.groupby('category')['exits'].cumsum()

# Calculate cumulative exits for each pool_size_label
validator_exits['size_cumulative'] = validator_exits.groupby('pool_size_label')['exits'].cumsum()
validator_exits

,validator_index,slot,exits,pool,category,pool_size_label,pool_cumulative,category_cumulative,size_cumulative
262450,20075,6816.0,1,Other Stakers,Unidentified,20-99,1,1.0,1
259189,4259,17248.0,1,Other Stakers,Liquid Staking,100+,2,1.0,1
72724,21574,17248.0,1,Other Stakers,Liquid Staking,100+,3,2.0,2
3242,4086,17344.0,1,Other Stakers,Liquid Staking,100+,4,3.0,3
74618,4100,17344.0,1,Other Stakers,Liquid Staking,100+,5,4.0,4
...,...,...,...,...,...,...,...,...,...
365332,605002,8985952.0,1,Rocketpool,Liquid Staking,100+,10924,67600.0,346654
215172,605000,8985952.0,1,Rocketpool,Liquid Staking,100+,10925,67601.0,346655
141044,604998,8985952.0,1,Rocketpool,Liquid Staking,100+,10926,67602.0,346656
254646,605036,8985952.0,1,Rocketpool,Liquid Staking,100+,10927,67603.0,346657


In [5]:
validator_activations = validator_metadata[['validator_index', 'activation_epoch']].dropna(subset=['activation_epoch'])
validator_activations['slot'] = validator_activations['activation_epoch'] * 32
validator_activations['activations'] = 1
validator_activations = validator_activations[validator_activations['slot'] <= 8986176]
validator_activations = validator_activations.drop(columns=['activation_epoch'])
validator_activations = pd.merge(validator_activations, validator_metadata[['validator_index', 'pool', 'category', 'pool_size_label']], on='validator_index', how='left')
validator_activations = validator_activations.sort_values('slot')
# Calculate cumulative activations for each pool
validator_activations['pool_cumulative'] = validator_activations.groupby('pool')['activations'].cumsum()

# Calculate cumulative activations for each category
validator_activations['category_cumulative'] = validator_activations.groupby('category')['activations'].cumsum()

# Calculate cumulative activations for each pool_size_label
validator_activations['size_cumulative'] = validator_activations.groupby('pool_size_label')['activations'].cumsum()
validator_activations

,validator_index,slot,activations,pool,category,pool_size_label,pool_cumulative,category_cumulative,size_cumulative
0,18846,0.0,1,Other Stakers,CEX,100+,1,1.0,1
222692,965,0.0,1,Other Stakers,NaN,6-19,2,NaN,1
222693,12110,0.0,1,Other Stakers,Unidentified,100+,3,1.0,2
222694,12109,0.0,1,Other Stakers,Unidentified,100+,4,2.0,3
222695,1313,0.0,1,Other Stakers,Unidentified,20-99,5,3.0,1
...,...,...,...,...,...,...,...,...,...
826874,1372201,8986176.0,1,Ether.Fi,Liquid Restaking,100+,34422,45826.0,1266732
1353715,1372208,8986176.0,1,Ether.Fi,Liquid Restaking,100+,34423,45827.0,1266733
827456,1372204,8986176.0,1,Ether.Fi,Liquid Restaking,100+,34424,45828.0,1266734
1227165,1372205,8986176.0,1,Ether.Fi,Liquid Restaking,100+,34425,45829.0,1266735


In [6]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool_size_label'] = np.nan
slots_df['exits'] = np.nan

# Get unique pool size labels
columns = validator_exits['pool_size_label'].unique()

# Assign pool size labels cyclically
slots_df['pool_size_label'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_exits by slot and pool_size_label, and sum the number of exits
exits_grouped = validator_exits.groupby(['slot', 'pool_size_label'])['exits'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
pool_size_exits_pivot = exits_grouped.pivot_table(index='slot', columns='pool_size_label', values='exits', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool_size_label', values='exits', dropna=False)
validator_exits_size = slots_pivot.combine_first(pool_size_exits_pivot).fillna(0)

validator_exits_size['total'] = validator_exits_size.sum(axis=1)

# Display the pivot table
validator_exits_size

pool_size_label,1,100+,2-5,20-99,6-19,total
slot,,,,,,
0.0,0.0,0.0,0.0,0.0,0.0,0.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
8986171.0,0.0,0.0,0.0,0.0,0.0,0.0
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool_size_label'] = np.nan
slots_df['activations'] = np.nan

# Get unique pool size labels
columns = validator_activations['pool_size_label'].unique()

# Assign pool size labels cyclically
slots_df['pool_size_label'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_activations by slot and pool_size_label, and sum the number of activations
activations_grouped = validator_activations.groupby(['slot', 'pool_size_label'])['activations'].sum().reset_index()

# Create the pivot table with the grouped and summed activations
pool_size_activations_pivot = activations_grouped.pivot_table(index='slot', columns='pool_size_label', values='activations', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool_size_label', values='activations', dropna=False)
validator_activations_size = slots_pivot.combine_first(pool_size_activations_pivot).fillna(0)

validator_activations_size['total'] = validator_activations_size.sum(axis=1)

# Display the pivot table
validator_activations_size

pool_size_label,1,100+,2-5,20-99,6-19,total
slot,,,,,,
0.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0
8986174.0,0.0,0.0,0.0,0.0,0.0,0.0


In [8]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['category'] = np.nan
slots_df['exits'] = np.nan

# Get unique pool size labels
columns = validator_exits['category'].unique()

# Assign pool size labels cyclically
slots_df['category'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_exits by slot and category, and sum the number of exits
exits_grouped = validator_exits.groupby(['slot', 'category'])['exits'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
category_exits_pivot = exits_grouped.pivot_table(index='slot', columns='category', values='exits', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='category', values='exits', dropna=False)
validator_exits_category = slots_pivot.combine_first(category_exits_pivot).fillna(0)

validator_exits_category['total'] = validator_exits_category.sum(axis=1)

# Display the pivot table
validator_exits_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,NaN,total
slot,,,,,,,,
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
8986171.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['category'] = np.nan
slots_df['activations'] = np.nan

# Get unique pool size labels
columns = validator_activations['category'].unique()

# Assign pool size labels cyclically
slots_df['category'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_activations by slot and category, and sum the number of activations
activations_grouped = validator_activations.groupby(['slot', 'category'])['activations'].sum().reset_index()

# Create the pivot table with the grouped and summed activations
category_activations_pivot = activations_grouped.pivot_table(index='slot', columns='category', values='activations', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='category', values='activations', dropna=False)
validator_activations_category = slots_pivot.combine_first(category_activations_pivot).fillna(0)

validator_activations_category['total'] = validator_activations_category.sum(axis=1)

# Display the pivot table
validator_activations_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,NaN,total
slot,,,,,,,,
0.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986174.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [10]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool'] = np.nan
slots_df['exits'] = np.nan

# Get unique pool size labels
columns = validator_exits['pool'].unique()

# Assign pool size labels cyclically
slots_df['pool'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_exits by slot and pool, and sum the number of exits
exits_grouped = validator_exits.groupby(['slot', 'pool'])['exits'].sum().reset_index()

# Create the pivot table with the grouped and summed exits
pool_exits_pivot = exits_grouped.pivot_table(index='slot', columns='pool', values='exits', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool', values='exits', dropna=False)
validator_exits_pool = slots_pivot.combine_first(pool_exits_pivot).fillna(0)

validator_exits_pool['total'] = validator_exits_pool.sum(axis=1)

# Display the pivot table
validator_exits_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
slot,,,,,,,,,,,,
0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8986171.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Create a slots frame for staked tables
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool'] = np.nan
slots_df['activations'] = np.nan

# Get unique pool size labels
columns = validator_activations['pool'].unique()

# Assign pool size labels cyclically
slots_df['pool'] = [columns[i % len(columns)] for i in range(len(slots_df))]

# Group validator_activations by slot and pool, and sum the number of activations
activations_grouped = validator_activations.groupby(['slot', 'pool'])['activations'].sum().reset_index()

# Create the pivot table with the grouped and summed activations
pool_activations_pivot = activations_grouped.pivot_table(index='slot', columns='pool', values='activations', fill_value=0)

# Merge the pivot table with slots_df
slots_pivot = slots_df.pivot_table(index='slot', columns='pool', values='activations', dropna=False)
validator_activations_pool = slots_pivot.combine_first(pool_activations_pivot).fillna(0)

validator_activations_pool['total'] = validator_activations_pool.sum(axis=1)

# Display the pivot table
validator_activations_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
slot,,,,,,,,,,,,
0.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8986172.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986173.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8986174.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [12]:
# Create slots DataFrame
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool_size_label'] = np.nan
slots_df['size_cumulative'] = np.nan

# Define the unique values in pool_size_label
columns = validator_exits['pool_size_label'].unique()

# Assign values to pool_size_label column cyclically
slots_df['pool_size_label'] = [columns[i % len(columns)] for i in range(len(slots_df))]

slots_pivot = slots_df.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative', dropna=False)

# Create pivot table for slots
validator_activations_raw = validator_activations.drop_duplicates(subset=['slot', 'pool_size_label'], keep='last')
validator_exits_raw = validator_exits.drop_duplicates(subset=['slot', 'pool_size_label'], keep='last')

# Create pivot table for activations
validator_activations_raw = validator_activations_raw.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative')
validator_exits_raw = validator_exits_raw.pivot_table(index='slot', columns='pool_size_label', values='size_cumulative')

# Filter validator_activations_raw to only include slots present in slots_pivot
validator_activations_filtered = validator_activations_raw[validator_activations_raw.index.isin(slots_pivot.index)]
validator_exits_filtered = validator_exits_raw[validator_exits_raw.index.isin(slots_pivot.index)]

# Combine filtered validator_activations_raw with slots_pivot
validator_activations_pivot = slots_pivot.combine_first(validator_activations_filtered).ffill()
validator_exits_pivot = slots_pivot.combine_first(validator_exits_filtered).ffill()
validator_exits_pivot = validator_exits_pivot.fillna(0)

active_validators_size = validator_activations_pivot.subtract(validator_exits_pivot, fill_value=0)

active_validators_size['total'] = active_validators_size.sum(axis=1)

active_validators_size

pool_size_label,1,100+,2-5,20-99,6-19,total
slot,,,,,,
0.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
1.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
2.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
3.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
4.0,1020.0,13599.0,1049.0,3568.0,1827.0,21063.0
...,...,...,...,...,...,...
8986171.0,9845.0,920070.0,9926.0,44358.0,18083.0,1002282.0
8986172.0,9845.0,920070.0,9926.0,44358.0,18083.0,1002282.0
8986173.0,9845.0,920070.0,9926.0,44358.0,18083.0,1002282.0


In [13]:
# Create slots DataFrame
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['category'] = np.nan
slots_df['category_cumulative'] = np.nan

# Define the unique values in category
columns = validator_exits['category'].unique()

# Assign values to category column cyclically
slots_df['category'] = [columns[i % len(columns)] for i in range(len(slots_df))]

slots_pivot = slots_df.pivot_table(index='slot', columns='category', values='category_cumulative', dropna=False)

# Create pivot table for slots
validator_activations_raw = validator_activations.drop_duplicates(subset=['slot', 'category'], keep='last')
validator_exits_raw = validator_exits.drop_duplicates(subset=['slot', 'category'], keep='last')

# Create pivot table for activations
validator_activations_raw = validator_activations_raw.pivot_table(index='slot', columns='category', values='category_cumulative')
validator_exits_raw = validator_exits_raw.pivot_table(index='slot', columns='category', values='category_cumulative')

# Filter validator_activations_raw to only include slots present in slots_pivot
validator_activations_filtered = validator_activations_raw[validator_activations_raw.index.isin(slots_pivot.index)]
validator_exits_filtered = validator_exits_raw[validator_exits_raw.index.isin(slots_pivot.index)]

# Combine filtered validator_activations_raw with slots_pivot
validator_activations_pivot = slots_pivot.combine_first(validator_activations_filtered).ffill()
validator_exits_pivot = slots_pivot.combine_first(validator_exits_filtered).ffill()
validator_exits_pivot = validator_exits_pivot.fillna(0)

active_validators_category = validator_activations_pivot.subtract(validator_exits_pivot, fill_value=0)

active_validators_category['total'] = active_validators_category.sum(axis=1)

active_validators_category

category,CEX,Liquid Restaking,Liquid Staking,Solo Stakers,Staking Pools,Unidentified,NaN,total
slot,,,,,,,,
0.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
1.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
2.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
3.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
4.0,2927.0,0.0,559.0,218.0,13.0,14105.0,0.0,17822.0
...,...,...,...,...,...,...,...,...
8986171.0,238161.0,44151.0,331421.0,630.0,18939.0,263500.0,0.0,896802.0
8986172.0,238161.0,44151.0,331421.0,630.0,18939.0,263500.0,0.0,896802.0
8986173.0,238161.0,44151.0,331421.0,630.0,18939.0,263500.0,0.0,896802.0


In [14]:
# Create slots DataFrame
slots_df = pd.DataFrame({'slot': range(0, 8986176)})
slots_df['pool'] = np.nan
slots_df['pool_cumulative'] = np.nan

# Define the unique values in pool
columns = validator_exits['pool'].unique()

# Assign values to pool column cyclically
slots_df['pool'] = [columns[i % len(columns)] for i in range(len(slots_df))]

slots_pivot = slots_df.pivot_table(index='slot', columns='pool', values='pool_cumulative', dropna=False)

# Create pivot table for slots
validator_activations_raw = validator_activations.drop_duplicates(subset=['slot', 'pool'], keep='last')
validator_exits_raw = validator_exits.drop_duplicates(subset=['slot', 'pool'], keep='last')

# Create pivot table for activations
validator_activations_raw = validator_activations_raw.pivot_table(index='slot', columns='pool', values='pool_cumulative')
validator_exits_raw = validator_exits_raw.pivot_table(index='slot', columns='pool', values='pool_cumulative')

# Filter validator_activations_raw to only include slots present in slots_pivot
validator_activations_filtered = validator_activations_raw[validator_activations_raw.index.isin(slots_pivot.index)]
validator_exits_filtered = validator_exits_raw[validator_exits_raw.index.isin(slots_pivot.index)]

# Combine filtered validator_activations_raw with slots_pivot
validator_activations_pivot = slots_pivot.combine_first(validator_activations_filtered).ffill()
validator_activations_pivot = validator_activations_pivot.fillna(0)
validator_exits_pivot = slots_pivot.combine_first(validator_exits_filtered).ffill()
validator_exits_pivot = validator_exits_pivot.fillna(0)

active_validators_pool = validator_activations_pivot.subtract(validator_exits_pivot, fill_value=0)

active_validators_pool['total'] = active_validators_pool.sum(axis=1)

active_validators_pool

pool,Binance,Bitcoin Suisse,Coinbase,Ether.Fi,Kraken,Ledger Live,Lido,Mantle,OKX,Other Stakers,Rocketpool,total
slot,,,,,,,,,,,,
0.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
1.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
2.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
3.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
4.0,1.0,2876.0,0.0,0.0,0.0,13.0,0.0,0.0,0.0,18153.0,20.0,21063.0
...,...,...,...,...,...,...,...,...,...,...,...,...
8986171.0,35167.0,19013.0,138336.0,33868.0,23748.0,14728.0,290691.0,14974.0,10877.0,396160.0,24720.0,1002282.0
8986172.0,35167.0,19013.0,138336.0,33868.0,23748.0,14728.0,290691.0,14974.0,10877.0,396160.0,24720.0,1002282.0
8986173.0,35167.0,19013.0,138336.0,33868.0,23748.0,14728.0,290691.0,14974.0,10877.0,396160.0,24720.0,1002282.0


In [15]:
validator_exits_size.to_csv('../int/validator_exits_size.csv')
validator_exits_category.to_csv('../int/validator_exits_category.csv')
validator_exits_pool.to_csv('../int/validator_exits_pool.csv')

In [16]:
active_validators_size.to_csv('../int/active_validators_size.csv')
active_validators_category.to_csv('../int/active_validators_category.csv')
active_validators_pool.to_csv('../int/active_validators_pool.csv')